In [1]:
import sys
sys.path.append('/Users/aidanmorson/Desktop/analysis/parkinsons/analysis')

In [5]:
from analysis.position_sd import get_neuron_positions, compute_2d_histogram, compute_2d_kde
from analysis.stats_sd import reduce_dimensionality
from load_data.load_npz import load_spikedata
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
import os
import numpy as np


In [3]:
sd24578_49 = load_spikedata('/Users/aidanmorson/Desktop/analysis/parkinsons/data-by-day/d49/24578_D49-6.zip')
sd24578_60 = load_spikedata('/Users/aidanmorson/Desktop/analysis/parkinsons/data-by-day/d60/24578_D60+6.zip')
sd24481_49 = load_spikedata('/Users/aidanmorson/Desktop/analysis/parkinsons/data-by-day/d49/24481_D49-6.zip')
sd24481_60 = load_spikedata('/Users/aidanmorson/Desktop/analysis/parkinsons/data-by-day/d60/24481_D60+6.zip')

In [4]:
positions_24578_49 = get_neuron_positions(sd24578_49)
positions_24578_60 = get_neuron_positions(sd24578_60)
positions_24481_49 = get_neuron_positions(sd24481_49)
positions_24481_60 = get_neuron_positions(sd24481_60)

In [16]:
print(np.min(positions_24578_49, axis=0), np.max(positions_24578_49, axis=0))
print(np.min(positions_24578_60, axis=0), np.max(positions_24578_60, axis=0))
print(np.min(positions_24481_49, axis=0), np.max(positions_24481_49, axis=0))
print(np.min(positions_24481_60, axis=0), np.max(positions_24481_60, axis=0))

[472.5 560. ] [1855.  1802.5]
[752.5 525. ] [1785.  1207.5]
[1732.5  455. ] [3062.5 1785. ]
[1347.5  262.5] [3132.5 1942.5]


In [17]:
density_24578_49, x24578_49, y24578_49 = compute_2d_histogram(positions_24578_49)
density_24578_60, x24578_60, y24578_60 = compute_2d_histogram(positions_24578_60)
density_24481_49, x24481_49, y_24481_49 = compute_2d_histogram(positions_24481_49)
density_24481_60, x24481_60, y24481_60 = compute_2d_histogram(positions_24481_60)

In [48]:
def plot_neuron_density(density, xedges, yedges, title="Chip: Day", sigma=0, output_path=None, filename=None):
    """
    Plot a 2D density map (histogram) of neuron positions.
    
    Parameters:
      density : np.ndarray
          2D array of shape (nx, ny) representing the neuron count in each bin (or KDE value).
      xedges, yedges : np.ndarray
          Arrays of bin edges from histogram2d (or xgrid, ygrid if using KDE).
      title : str
          Chip and day identifier for data.
        smoothing : bool
          If True, plot a smoothed KDE instead of a histogram.
      
    """
    plt.figure(figsize=(6,5))
    
    if sigma > 0:
        density = gaussian_filter(density, sigma=sigma)

    # For a histogram: xedges and yedges are the bin edges
    extent = [xedges[0], xedges[-1], yedges[0], yedges[-1]]
    
    # We can transpose density if needed to align axes. 
    # For histogram2d, density is shape (nx, ny), x is axis=0, y is axis=1.
    # If you find it's flipped, do density.T.
    
    img = plt.imshow(density.T, origin='lower', cmap='hot', 
               extent=extent, aspect='auto', vmin=0, vmax=4)
    

    cbar = plt.colorbar(img)
    cbar.set_label("Neuron Density (count)" if density.dtype.kind in ('i','u','f') else "Density")

    
    plt.title(f"Density of Neurons: {title}")
    plt.xlabel("Horizontal Position (µm)")
    plt.ylabel("Veritcal Position (µm)")
    #plt.xlim(450, 3150)
    #plt.ylim(250, 1950)
    plt.tight_layout()
    if output_path:
        # If output_path is a directory, save with a default filename.
        if os.path.isdir(output_path):
            save_path = os.path.join(output_path, f"density-smoothed-{filename}.png")
        else:
            save_path = output_path
        plt.savefig(save_path)
        plt.close()
    else:
        plt.show()


In [47]:
def raw_footprint(sd, output_path=None, title=None):
    """
    Plot a basic footprint of neuron positions.

    Parameters:
        sd : SpikeData
            A SpikeData object with neuron positions stored in sd.neuron_data.
        output_path : str, optional
            Directory in which to save the plot. If None, the plot is returned.

    Returns:
        fig, ax : Matplotlib figure and axes objects.
    """
    # Use the first element if sd.neuron_data is a list.
    positions = get_neuron_positions(sd)
    
    fig, ax = plt.subplots(figsize=(8,8))
    ax.scatter(positions[:, 0], positions[:, 1], c='blue', s=50)
    ax.set_xlabel("Horizontal Position (µm)")
    ax.set_ylabel("Vertical Position (µm)")
    ax.set_xlim(450, 3150)
    ax.set_ylim(250, 1950)
    ax.set_title(f"Footprint: {title}")
    
    if output_path:
        os.makedirs(output_path, exist_ok=True)
        plt.savefig(os.path.join(output_path, f"footprint_{title}.png"))
        plt.close(fig)
    return fig, ax

In [22]:
raw_footprint(sd24578_49, output_path='/Users/aidanmorson/Desktop/analysis/parkinsons/density/', title="24578_49")
raw_footprint(sd24578_60, output_path='/Users/aidanmorson/Desktop/analysis/parkinsons/density/', title="24578_60")
raw_footprint(sd24481_49, output_path='/Users/aidanmorson/Desktop/analysis/parkinsons/density/', title="24481_49")
raw_footprint(sd24481_60, output_path='/Users/aidanmorson/Desktop/analysis/parkinsons/density/', title="24481_60")

(<Figure size 800x800 with 1 Axes>,
 <Axes: title={'center': 'Footprint: 24481_60'}, xlabel='Horizontal Position (µm)', ylabel='Vertical Position (µm)'>)

In [49]:
plot_neuron_density(density_24578_49, x24578_49, y24578_49, title="24578, Day 49", sigma=1, output_path='/Users/aidanmorson/Desktop/analysis/parkinsons/density/', filename='24578_49')
plot_neuron_density(density_24578_60, x24578_60, y24578_60, title="24578, Day 60", sigma=1, output_path='/Users/aidanmorson/Desktop/analysis/parkinsons/density/', filename='24578_60')
plot_neuron_density(density_24481_49, x24481_49, y_24481_49, title="24481, Day 49", sigma=1, output_path='/Users/aidanmorson/Desktop/analysis/parkinsons/density/', filename='24481_49')
plot_neuron_density(density_24481_60, x24481_60, y24481_60, title="24481, Day 60", sigma=1, output_path='/Users/aidanmorson/Desktop/analysis/parkinsons/density/', filename='24481_60')

In [38]:
import numpy as np
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

def pca_on_matrix(matrix):
    """
    Perform PCA on an NxN matrix, treating each row as a sample of dimension N.
    Returns the PCA object (so you can access explained_variance_, etc.).
    """
    pca = PCA(n_components=None)  # keep all components
    pca.fit(matrix)  # shape (N, N)
    return pca


In [25]:
pca_24578_49 = pca_on_matrix(density_24578_49)
pca_24578_60 = pca_on_matrix(density_24578_60)
pca_24481_49 = pca_on_matrix(density_24481_49)
pca_24481_60 = pca_on_matrix(density_24481_60)

In [57]:
def plot_pca_combined(pca_list, pca_names_list, colors_list, title=None, output_path=None):
    """
    Plot the eigenvalue spectrum for multiple PCA objects on the same plot.
    """
    plt.figure(figsize=(8,6))
    for i, pca in enumerate(pca_list):
        dataset_name=pca_names_list[i]
        color=colors_list[i]
        eigenvals = pca.explained_variance_
        plt.plot(range(1, len(eigenvals)+1), eigenvals, 'o-', label=f"{dataset_name}", color=color)
    plt.xscale('linear')
    plt.yscale('log')
    plt.xlabel("Eigenvector Index")
    plt.ylabel("Eigenvalue (log scale)")
    plt.title("Eigenvalue Spectrum of Neuron Density")
    plt.legend()
    plt.tight_layout()
    if output_path:
        os.makedirs(output_path, exist_ok=True)
        plt.savefig(os.path.join(output_path, f"pca_{title}.png"))
        plt.close()
    else:
        plt.show()

In [72]:
full_colors = ["royalblue", "cyan", "green", "lime"]

In [73]:
pca_list = [pca_24578_49, pca_24578_60, pca_24481_49, pca_24481_60]
pca_names_list = ["24578: D49", "24578: D60", "24481: D49", "24481: D60"]


In [74]:
plot_pca_combined(pca_list, pca_names_list=pca_names_list, colors_list=full_colors, title="full", output_path='/Users/aidanmorson/Desktop/analysis/parkinsons/density/')

In [69]:
def summarize_density_arrays(density_list, colors, dataset_names):
    """
    For each 2D density array in density_list, compute summary statistics:
      - sum, mean, median, variance, std, min, max
    Then return a list of dictionaries with those stats.

    Parameters:
      density_list : list of np.ndarray
          Each element is a 2D array (e.g., histogram counts or density).
      dataset_names : list of str
          Names for each dataset (same length as density_list).

    Returns:
      summary_list : list of dict
         Each dict contains the dataset name plus the summary stats.
         Example:
         [
           {
             "dataset": "Dataset1",
             "bin_count": 2500,  # e.g. if the 2D array was shape (50, 50)
             "sum": ...,
             "mean": ...,
             "median": ...,
             "variance": ...,
             "std": ...,
             "min": ...,
             "max": ...
           },
           ...
         ]
    """
    summary_list = []
    for i, density in enumerate(density_list):
        dataset_name = dataset_names[i] if i < len(dataset_names) else f"Dataset_{i+1}"
        color = colors[i]

        # Flatten the 2D array so we can compute standard 1D stats
        flat_vals = density.ravel()
        # Optionally ignore zero bins if you only care about bins that have >0
        # flat_vals = flat_vals[flat_vals > 0]

        # Basic stats
        summary_dict = {
            "dataset": dataset_name,
            "color": color,
            "bin_count": len(flat_vals),
            "sum": float(np.sum(flat_vals)),
            "mean": float(np.mean(flat_vals)),
            "median": float(np.median(flat_vals)),
            "variance": float(np.var(flat_vals)),
            "std": float(np.std(flat_vals)),
            "min": float(np.min(flat_vals)),
            "max": float(np.max(flat_vals))
        }
        summary_list.append(summary_dict)
    return summary_list

In [70]:
def plot_density_variance_bar(summaries, output_path=None):
    """
    Given the list of summary dicts from summarize_density_arrays(),
    plot a bar chart of the variance of bin values vs. dataset.
    
    Parameters:
      summaries : list of dict
          Output from summarize_density_arrays().
      output_path : str, optional
          If provided, the figure is saved. Otherwise shown interactively.
    """
    dataset_labels = [s["dataset"] for s in summaries]
    colors = [s["color"] for s in summaries]
    variances = [s["variance"] for s in summaries]
    
    plt.figure(figsize=(8,6))
    x_pos = np.arange(len(dataset_labels))
    plt.bar(x_pos, variances, color=colors)
    plt.xticks(x_pos, dataset_labels, rotation=45, ha='right')
    plt.ylabel("Variance of Bin Values")
    plt.title("Variance of 2D Density Arrays")
    plt.tight_layout()

    if output_path:
        if os.path.isdir(output_path):
            save_path = os.path.join(output_path, "density_variance_bar.png")
        else:
            save_path = output_path
        plt.savefig(save_path)
        plt.close()
        print(f"Saved variance bar chart to: {save_path}")
    else:
        plt.show()


In [71]:
density_list = [density_24578_49, density_24578_60, density_24481_49, density_24481_60]
dataset_names = ["24578: D49", "24578: D60", "24481: D49", "24481: D60"]
summaries = summarize_density_arrays(density_list, full_colors, dataset_names)
plot_density_variance_bar(summaries, output_path='/Users/aidanmorson/Desktop/analysis/parkinsons/density/')

Saved variance bar chart to: /Users/aidanmorson/Desktop/analysis/parkinsons/density/density_variance_bar.png
